In [ ]:
# Hyperparameters and paths
DATA_DIR = r"/home/c/choton/beemachine/datasets/Beemachine_Partwhole_v5/"
NUM_PARTS = 4
CLASSIFIER_NAME = "deit_base_distilled_patch16_224.fb_in1k"
SEGMENTATION_NAME = "deeplabv3plus"
SEGMENTATION_ENCODER = "resnext50_32x4d"
SEGMENTATION_WEIGHTS = r"/home/c/choton/beemachine/codes/AG_vision_2026/2_segmentation/Beemachine/deeplabv3+/lightning_logs/version_0/checkpoints/epoch=199-step=54400.ckpt"
FEATURE_SIZE = 937
IMAGE_SIZE = 224
DEVICE_ID = 0

# The classification models and their timm names
backbone_models = {
    "ConvNext_Nano": "convnext_nano.in12k",
    "DeiT": "deit_base_distilled_patch16_224.fb_in1k",
    # "EfficientNetV2L": "tf_efficientnetv2_l.in21k",
    "EfficientNetV2M": "efficientnetv2_rw_m.agc_in1k",
    "EfficientNetV2S": "efficientnetv2_rw_s.ra2_in1k",
    # "EVA02_Large":"eva02_large_patch14_224.mim_in22k",
    "InceptionNext_Tiny": "inception_next_tiny.sail_in1k",
    "ResNet101": "resnet101.a1_in1k",
    "SeResNext101": "seresnext101_32x4d.gluon_in1k",
    "SWIN_Classifier": "swin_base_patch4_window7_224.ms_in1k",
}

# The weights of the PADC models
padc_weights = {
    "ConvNext_Nano": r"/home/c/choton/beemachine/codes/AG_vision_2026/PADC/Beemachine/PADC_Part/PADC_ab_convnext_nano_fixed_logs/checkpoints/best_model.pth",
    "DeiT": r"/home/c/choton/beemachine/codes/AG_vision_2026/PADC/Beemachine/PADC_Ablation/padc_ab_deit_fixed_logs/checkpoints/best_model.pth",
    # "EfficientNetV2L": "tf_efficientnetv2_l.in21k",
    "EfficientNetV2M": "/home/c/choton/beemachine/codes/AG_vision_2026/PADC/Beemachine/PADC_Ablation/padc_arch2_effnet_v2m_fixed_logs/checkpoints/best_model.pth",
    "EfficientNetV2S": "/home/c/choton/beemachine/codes/AG_vision_2026/PADC/Beemachine/PADC_Ablation/padc_arch2_effnets_fixed_logs/checkpoints/best_model.pth",
    # "EVA02_Large":"eva02_large_patch14_224.mim_in22k",
    "InceptionNext_Tiny": "/home/c/choton/beemachine/codes/AG_vision_2026/PADC/Beemachine/PADC_Ablation/padc_arch2_inceptionnext_tiny_fixed_logs/checkpoints/best_model.pth",
    "ResNet101": "/home/c/choton/beemachine/codes/AG_vision_2026/PADC/Beemachine/PADC_Ablation/padc_arch2_resnet101_fixed_logs/checkpoints/best_model.pth",
    "SeResNext101": "/home/c/choton/beemachine/codes/AG_vision_2026/PADC/Beemachine/PADC_Ablation/padc_arch2_se_resnext101_fixed_logs/checkpoints/best_model.pth",
    "SWIN_Classifier": "/home/c/choton/beemachine/codes/AG_vision_2026/PADC/Beemachine/PADC_Ablation/padc_arch2_swin_fixed_logs/checkpoints/best_model.pth",
}

# The weights of the baseline models
baseline_weights = {
    "ConvNext_Nano": r"/home/c/choton/beemachine/codes/AG_vision_2026/1_classification/Beemachine/convnext_nano.in12k_timm_new_dataset_logs/checkpoints/best_model.pth",
    "DeiT": r"/home/c/choton/beemachine/codes/AG_vision_2026/1_classification/Beemachine/deit_base_distilled_patch16_224.fb_in1k_timm_new_dataset_logs/checkpoints/best_model.pth",
    # "EfficientNetV2L": "tf_efficientnetv2_l.in21k",
    "EfficientNetV2M": r"/home/c/choton/beemachine/codes/AG_vision_2026/1_classification/Beemachine/efficientnetv2_rw_m.agc_in1k_timm_logs/checkpoints/best_model.pth",
    "EfficientNetV2S": r"/home/c/choton/beemachine/codes/AG_vision_2026/1_classification/Beemachine/efficientnetv2_rw_s.ra2_in1k_timm_v1_logs/checkpoints/best_model.pth",
    # "EVA02_Large":"eva02_large_patch14_224.mim_in22k",
    "InceptionNext_Tiny": r"/home/c/choton/beemachine/codes/AG_vision_2026/1_classification/Beemachine/inception_next_tiny.sail_in1k_timm_new_dataset_logs/checkpoints/best_model.pth",
    "ResNet101": r"/home/c/choton/beemachine/codes/AG_vision_2026/1_classification/Beemachine/resnet101.a1_in1k_timm_new_dataset_logs/checkpoints/best_model.pth",
    "SeResNext101": r"/home/c/choton/beemachine/codes/AG_vision_2026/1_classification/Beemachine/seresnext101_32x4d.gluon_in1k_timm_new_dataset_logs/checkpoints/best_model.pth",
    "SWIN_Classifier": r"/home/c/choton/beemachine/codes/AG_vision_2026/1_classification/Beemachine/swin_base_patch4_window7_224.ms_in1k_timm_new_dataset_logs/checkpoints/best_model.pth",
}

classifier_names = list(padc_weights.keys())

In [ ]:
# ---------------------------------------
# 🔥 Plot FinerCAM for ALL backbones (Grid Layout)
# ---------------------------------------

image_no = 0
image, lbl, img_path = test_dataset[image_no]
num_models = len(classifier_names)
cols = 3
rows = (num_models + cols - 1) // cols  # Ceiling division to find required rows

fig, axes = plt.subplots(rows, cols, figsize=(4 * cols, 4 * rows))
if num_models > 1: # Ensure axes is always a flat array even if there's only 1 row or 1 model
    axes = axes.flatten()
else:
    axes = [axes]

for idx, classifier_name in enumerate(classifier_names):
    print(f"\n===== Processing {classifier_name} =====")
    
    # Load model    
    device = torch.device(f"cuda:{DEVICE_ID}")
    state_dict = torch.load(padc_weights[classifier_name], map_location=device)
    model = DeepShapeFusionModel(
        num_classes=num_classes,
        classifier_name=backbone_models[classifier_name],
        shape_embed_dim=512
    )
    model.load_state_dict(state_dict)
    model.to(device)
    model.eval()
    backbone = model.backbone

    # Automatically choose target layer
    reshape_fn = None
    model_name = backbone_models[classifier_name].lower()
    # ---- ViT / DeiT ----
    if "deit" in model_name or "vit" in model_name:
        target_layers = [backbone.blocks[-1].norm1]
        reshape_fn = reshape_transform_deit
    # ---- Swin ----
    elif "swin" in model_name:
        target_layers = [backbone.layers[-1]] #.blocks[-1].norm1]
        reshape_fn = reshape_transform_swin
    # ---- ConvNeXt ----
    elif "convnext" in model_name:
        target_layers = [backbone.stages[-1]] #.blocks[-1].norm]
    # ---- InceptionNext ----
    elif "inception_next" in model_name:
        target_layers = [backbone.stages[-1]] #.blocks[-1].norm]
    # ---- EfficientNet ----
    elif "efficientnet" in model_name:
        target_layers = [backbone.conv_head]
    # ---- ResNet / SeResNext ----
    elif "resnet" in model_name or "seresnext" in model_name:
        target_layers = [backbone.layer4[-1]]
    else:
        raise ValueError(f"Unknown architecture type for {classifier_name}")

    # Create FinerCAM
    cam = FinerCAM(
        model=model,   # FULL PADC model
        target_layers=target_layers,
        reshape_transform=reshape_fn
    )

    # Plot
    plot_cam(model=model, img_path=img_path, label=lbl, cam_method=cam, transform=test_dataset.transform, ax=axes[idx])
    axes[idx].set_title(classifier_name, fontsize=10)

# --- Clean up empty subplots ---
# If num_models is not a multiple of 3, hide the remaining empty axes
for j in range(idx + 1, len(axes)):
    axes[j].axis('off')
plt.tight_layout()
plt.show()